# InboxWorld Finale Notebook

This Colab notebook runs the judge-facing proof for **InboxWorld**, a multi-agent email triage environment with hidden urgency, delayed penalties, and adaptive decision policies.

What it demonstrates:
- OpenEnv-style environment setup
- Baseline vs multi-agent policy evaluation
- Adaptive environment simulation with generated artifacts
- RL-style transition buffer collection
- A live custom-email inference example
- Optional minimal Hugging Face TRL fine-tuning scaffold

## 1. Setup

Run this cell first. If the notebook is opened directly in Colab, it clones the Hugging Face Space repo. If you already uploaded the project folder, it uses the local folder.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://huggingface.co/spaces/RoopanshSaxena/InboxWorld"
PROJECT_DIR = Path("/content/InboxWorld")

def run(cmd):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

if not (Path.cwd() / "src" / "inboxworld").exists():
    if not PROJECT_DIR.exists():
        run(["git", "clone", REPO_URL, PROJECT_DIR])
    os.chdir(PROJECT_DIR)
else:
    PROJECT_DIR = Path.cwd()

print("Project directory:", PROJECT_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Setup complete.")

## 2. Smoke Test

This verifies that the package imports and the environment entry points are available.

In [ ]:
import inboxworld

print("InboxWorld exports:")
print(", ".join(inboxworld.__all__))

## 3. Static Benchmark: Baseline vs Multi-Agent

The baseline is intentionally simple. The multi-agent policy uses classifier, priority, responder, and supervisor stages.

In [ ]:
from pprint import pprint

from inboxworld import (
    InboxWorldEnv,
    baseline_policy,
    default_scenarios,
    evaluate_policy_set,
    multi_agent_policy,
    run_episode,
)

scenarios = default_scenarios()

baseline_summary = run_episode(InboxWorldEnv(scenarios), baseline_policy)
multi_agent_summary = run_episode(InboxWorldEnv(scenarios), multi_agent_policy)

print("Baseline summary")
pprint(baseline_summary)
print("\nMulti-agent summary")
pprint(multi_agent_summary)

benchmark = evaluate_policy_set(
    scenarios,
    {"baseline": baseline_policy, "multi_agent": multi_agent_policy},
)

print("\nAggregate comparison")
pprint({name: result["aggregate"] for name, result in benchmark.items()})

## 4. Adaptive Environment Simulation

This is the main finale evidence: a naive/greedy policy misses delayed consequences, while the multi-agent policy handles urgency and deadline pressure.

In [ ]:
from pathlib import Path
from IPython.display import SVG, display

from inboxworld import (
    EmailEnvironment,
    GreedyBaselinePolicy,
    MultiAgentEmailPolicy,
    RandomNaivePolicy,
    dynamic_environment_configs,
    run_learning_curve,
    run_simulation,
)

artifacts_dir = PROJECT_DIR / "artifacts" / "adaptive_environment"
env_factory = lambda: EmailEnvironment(dynamic_environment_configs())

baseline_results = run_simulation(
    EmailEnvironment(dynamic_environment_configs()),
    GreedyBaselinePolicy(),
    episodes=8,
    output_dir=artifacts_dir / "baseline",
)

adaptive_results = run_simulation(
    EmailEnvironment(dynamic_environment_configs()),
    MultiAgentEmailPolicy(),
    episodes=8,
    output_dir=artifacts_dir / "multi_agent",
)

learning_curve = run_learning_curve(
    env_factory=env_factory,
    naive_agent=RandomNaivePolicy(seed=17),
    improved_agent=MultiAgentEmailPolicy(),
    episodes=12,
    switch_episode=6,
    output_dir=artifacts_dir / "learning_curve",
)

print("Baseline aggregate")
pprint(baseline_results["aggregate"])
print("\nAdaptive multi-agent aggregate")
pprint(adaptive_results["aggregate"])
print("\nLearning curve aggregate")
pprint(learning_curve["aggregate"])

display(SVG(filename=str(artifacts_dir / "learning_curve" / "learning_curve.svg")))

## 5. Collect RL-Style Transition Buffer

This creates prompt/action/reward records that can be used for RLVR or supervised warm-start training.

In [ ]:
import json

from inboxworld import EmailEnvironment, collect_episode_transitions, dynamic_environment_configs

env = EmailEnvironment(dynamic_environment_configs())
transitions = collect_episode_transitions(env, episodes=3)

output_dir = PROJECT_DIR / "artifacts" / "training"
output_dir.mkdir(parents=True, exist_ok=True)
transition_path = output_dir / "transition_buffer.json"
transition_path.write_text(json.dumps(transitions, indent=2), encoding="utf-8")

print(f"Collected {len(transitions)} transitions")
print("Saved to", transition_path)
print("\nSample transition:")
pprint(transitions[0])

## 6. Live Custom Email Test

Change the sender, subject, body, importance, and urgency flags to show judges how the policy reacts to new inbox events.

In [ ]:
from dataclasses import asdict

from inboxworld import MultiAgentEmailPolicy
from inboxworld.types import InboxEmail

policy = MultiAgentEmailPolicy()

email = InboxEmail(
    email_id="judge-live-1",
    sender="Avery Chen",
    sender_importance="client",
    subject="Contract blocked today",
    body="The contract is blocked and we need an answer today before finance escalates.",
    priority="unknown",
    urgency=True,
    visible_urgency=True,
    deadline_step=1,
    hidden_intent="contract_risk",
    expected_action="generate_reply",
    expected_tone="professional",
    thread_id="judge-thread",
    tags=["contract"],
)

action = policy.act({"emails": [email], "time_step": 0, "config_id": "judge-demo"})
pprint(asdict(action))

## 7. Optional: Minimal HF TRL Training Run

This cell is off by default because it downloads a model and can take longer. Set `RUN_TRAINING = True` when you have GPU runtime enabled in Colab.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    run([sys.executable, "-m", "pip", "install", "-q", "datasets", "transformers", "trl", "accelerate"])
    run([sys.executable, "scripts/minimal_trl_train.py"])
else:
    print("Skipped training. Set RUN_TRAINING = True to run the minimal TRL scaffold.")

## Finale Takeaway

InboxWorld demonstrates that email triage is not just classification. The agent has to reason over sender importance, hidden urgency, deadlines, delayed penalties, and escalation risk. The multi-agent policy is designed to make those intermediate decisions inspectable for judges.